# M1 — Knowledge Gap Classifier (DeBERTa-v3-base)
Target: **F1 (micro) >= 0.85**. See `docs/phases/MENTORA_Phase4_Model_Training.md` §4.

**Two real adaptations from the plan doc, disclosed here rather than silently
changed:**
1. **Single-label, not multi-label.** The doc assumes a pipe-separated
   multi-topic `topic` column. The real data has exactly one `topic` per
   question, so this trains a standard single-label multi-class classifier
   (softmax, not sigmoid) instead of `problem_type="multi_label_classification"`.
2. **Topic labels for the 1777 MDCAT-derived questions are heuristic**
   (keyword-matched, not manually tagged — see
   `datasets/m1_gap_classifier/tag_topics.py` and `mentora_data/SOURCES.md`).
   67% of those rows fell back to a generic "General <Subject>" bucket because
   no keyword matched. The 378 Math questions have real topic/subtopic labels
   from their source dataset. **Treat the reported F1 as a first-pass signal,
   not a final number** — manually reviewing/improving the MDCAT topic tags
   would likely raise it.

## Setup — mount Drive, confirm GPU, install deps
Same first 3 cells in every Mentora training notebook. See `docs/phases/MENTORA_Phase4_Model_Training.md` §0.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.

In [ ]:
import os
DATA = '/content/drive/MyDrive/mentora_data'
MODELS = '/content/drive/MyDrive/mentora_models'
for m in ['model1_gap_classifier', 'model2_question_generator',
          'model3_trajectory_predictor', 'model4_career_matcher', 'model5_cv_ner']:
    os.makedirs(f'{MODELS}/{m}', exist_ok=True)

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate

## Load and tokenize

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd, numpy as np

df = pd.read_csv(f'{DATA}/processed/m1/question_bank_final.csv')
df = df.dropna(subset=['question_text', 'topic'])
topics = sorted(df['topic'].unique())
NUM_TOPICS = len(topics)
topic_to_idx = {t: i for i, t in enumerate(topics)}
print(f"{NUM_TOPICS} distinct topics, {len(df)} questions")
print(df['topic'].value_counts())

df['label'] = df['topic'].map(topic_to_idx)

tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
def tokenize(batch):
    return tokenizer(batch['question_text'], truncation=True, padding='max_length', max_length=128)

ds = Dataset.from_pandas(df[['question_text', 'label']])
ds = ds.train_test_split(test_size=0.15, seed=42)
ds = ds.map(tokenize, batched=True)

## Model + metrics

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'microsoft/deberta-v3-base', num_labels=NUM_TOPICS
)

from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }

## Train — epoch checkpointing + auto-resume

In [ ]:
OUT_DIR = f'{MODELS}/model1_gap_classifier'

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    fp16=True,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=ds['train'], eval_dataset=ds['test'],
    compute_metrics=compute_metrics,
)

import glob
existing_checkpoints = glob.glob(f'{OUT_DIR}/checkpoint-*')
resume = sorted(existing_checkpoints)[-1] if existing_checkpoints else None
trainer.train(resume_from_checkpoint=resume)

## Save final model

In [ ]:
trainer.save_model(f'{OUT_DIR}/final')
tokenizer.save_pretrained(f'{OUT_DIR}/final')
import json
with open(f'{OUT_DIR}/final/topic_labels.json', 'w') as f:
    json.dump(topics, f)  # save the label->topic mapping alongside the model — needed for inference
print(trainer.evaluate())

## If you miss the target (F1 micro < 0.85)
- This is very likely the heuristic-topic-label noise described above, not a
  training problem — check per-class F1 and see if the "General <Subject>"
  fallback classes (which mix unrelated questions together) are dragging the
  average down specifically
- If time allows, hand-correcting even just the MDCAT rows' topics (or
  merging the noisiest "General X" buckets back into subject-level) is the
  highest-leverage fix, more so than hyperparameter tuning
- Otherwise: sweep the classification threshold isn't applicable here (single-label
  softmax, not multi-label) — try `learning_rate=1e-5` or `num_train_epochs=8`
  if it looks like underfitting rather than label noise